# 04b — Outcome Prediction: Full Traces (k = all)

Companion to `04_prediction_outcome.ipynb` which sweeps over prefix lengths.  
Here we train on **complete traces only** (`k='all'`) using the same strict  
chronological 60/20/20 split and the same three encodings (Boolean, Frequency,  
Succession). Results are directly comparable to the `k=all` row in NB04.

Models: **Random Forest**, **XGBoost**, **LightGBM** (+ SMOTE-NC on training data).

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import pm4py as pm
import random
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score,
    precision_recall_curve,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb

# ── SMOTE-NC toggle ───────────────────────────────────────────────────────────
USE_SMOTENC = True

if USE_SMOTENC:
    try:
        from imblearn.over_sampling import SMOTENC, SMOTE
    except ImportError as _e:
        raise ImportError(f'Install imbalanced-learn: pip install imbalanced-learn  ({_e})') from _e

## 2. Load Data

In [2]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv(
    'data/event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp'],
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Events     : {len(df):,}')
print(f'Cases      : {df["Case_id"].nunique():,}')
print(f'Activities : {df["Step"].nunique()}')

Events     : 1,133,314
Cases      : 279,458
Activities : 58


In [3]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)
print('pm4py aliases added: case:concept:name | concept:name | time:timestamp')

pm4py aliases added: case:concept:name | concept:name | time:timestamp


## 3. Strict Chronological 60/20/20 Split

- **Train**: cases fully completed before the train cutoff.
- **Val**: cases starting after the train cutoff and ending before the val cutoff,
  plus *straddlers* (started pre-cutoff, ended in val window) — only post-cutoff events.
- **Test**: cases starting after the val cutoff, plus straddlers — only post-val events.

In [4]:
case_times = df.groupby('case:concept:name').agg(
    case_start=('time:timestamp', 'min'),
    case_end  =('time:timestamp', 'max'),
).sort_values('case_start')

n            = len(case_times)
train_cutoff = case_times.iloc[int(n * 0.60)]['case_start']
val_cutoff   = case_times.iloc[int(n * 0.80)]['case_start']

train_ids = case_times[case_times['case_end']   <  train_cutoff].index
val_ids   = case_times[(case_times['case_start'] >= train_cutoff) &
                        (case_times['case_end']   <  val_cutoff)].index
test_ids  = case_times[case_times['case_start'] >= val_cutoff].index

train_df = df[df['case:concept:name'].isin(train_ids)].copy()

straddle_tv = case_times[(case_times['case_start'] <  train_cutoff) &
                          (case_times['case_end']   >= train_cutoff) &
                          (case_times['case_end']   <  val_cutoff)].index
val_df = pd.concat([
    df[df['case:concept:name'].isin(val_ids)],
    df[df['case:concept:name'].isin(straddle_tv) & (df['time:timestamp'] >= train_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

straddle_vt = case_times[(case_times['case_start'] <  val_cutoff) &
                          (case_times['case_end']   >= val_cutoff)].index
test_df = pd.concat([
    df[df['case:concept:name'].isin(test_ids)],
    df[df['case:concept:name'].isin(straddle_vt) & (df['time:timestamp'] >= val_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f'Train cutoff : {train_cutoff.date()} | Val cutoff : {val_cutoff.date()}')
print(f'Train — {train_df["case:concept:name"].nunique():,} cases  |  {len(train_df):,} events')
print(f'Val   — {val_df["case:concept:name"].nunique():,} cases  |  {len(val_df):,} events')
print(f'Test  — {test_df["case:concept:name"].nunique():,} cases  |  {len(test_df):,} events')

Train cutoff : 2024-09-12 | Val cutoff : 2025-08-29
Train — 162,133 cases  |  691,878 events
Val   — 60,212 cases  |  195,485 events
Test  — 57,113 cases  |  206,146 events


## 4. Recruiting Agency Encoding

In [5]:
AGENCY_COL = 'Recruiting Agency'

def encode_agency(df_split, encoder=None, fit=False):
    """Label-encode Recruiting Agency. NaN → 'No Agency' (always index 0)."""
    s = df_split.groupby('case:concept:name')[AGENCY_COL].first().fillna('No Agency')
    if fit:
        encoder = LabelEncoder()
        encoder.fit(s)
    known = set(encoder.classes_)
    s_mapped = s.apply(lambda x: x if x in known else 'No Agency')
    return encoder.transform(s_mapped), encoder

train_agency_arr, agency_encoder = encode_agency(train_df, fit=True)
val_agency_arr,   _              = encode_agency(val_df,   encoder=agency_encoder)
test_agency_arr,  _              = encode_agency(test_df,  encoder=agency_encoder)

def _agency_series(df_split, arr):
    case_ids = df_split.groupby('case:concept:name')[AGENCY_COL].first().index
    return pd.Series(arr, index=case_ids, name='agency_enc')

train_agency = _agency_series(train_df, train_agency_arr)
val_agency   = _agency_series(val_df,   val_agency_arr)
test_agency  = _agency_series(test_df,  test_agency_arr)

n_agencies = len(agency_encoder.classes_)
print(f'Recruiting Agency: {n_agencies} unique values (incl. "No Agency")')
print(f'No-agency cases — train: {(train_agency == 0).sum():,} | '
      f'val: {(val_agency == 0).sum():,} | test: {(test_agency == 0).sum():,}')

Recruiting Agency: 61 unique values (incl. "No Agency")
No-agency cases — train: 1 | val: 0 | test: 0


## 5. Outcome Labels

In [6]:
y_train_series = (
    train_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
y_val_series = (
    val_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
y_test_series = (
    test_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
print(f'Train hire rate : {y_train_series.mean():.2%}  ({y_train_series.sum():,} / {len(y_train_series):,})')
print(f'Val   hire rate : {y_val_series.mean():.2%}  ({y_val_series.sum():,} / {len(y_val_series):,})')
print(f'Test  hire rate : {y_test_series.mean():.2%}  ({y_test_series.sum():,} / {len(y_test_series):,})')

Train hire rate : 2.42%  (3,921 / 162,133)
Val   hire rate : 1.54%  (925 / 60,212)
Test  hire rate : 1.35%  (771 / 57,113)


## 6. Remove Leaky Activities

Activities with a hire rate ≥ 0.85 on the training set are removed from all splits  
(they are post-outcome markers that would trivially reveal the label).

In [7]:
LEAKY_THRESHOLD = 0.85

act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)

print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

print(f'\nActivities remaining : {train_df["concept:name"].nunique()}')
print(f'Train events : {len(train_df):,}  |  Val events : {len(val_df):,}  |  Test events : {len(test_df):,}')

Leaky activities removed (16):
  1.000  Ready for Hire
  1.000  FurstPerson Integration
  0.997  Additional Offer Packet Document(s)
  0.990  Review Company and Office Guides
  0.985  Candidate Experience Survey
  0.983  Recruiting Process Survey
  0.974  Automatically Unpost Jobs
  0.967  Confirm No Manager Survey
  0.927  LATAM Standard Questionnaire (2024)
  0.924  Colombia Standard Questionnaire (2024)
  0.900  Additional Background / Reference Check
  0.891  Renegotiate Offer
  0.885  Make Background Check Decision
  0.885  Background Check (LATAM)
  0.884  Set Background Check Status
  0.851  Consolidated Approval by Initiator

Activities remaining : 35
Train events : 667,960  |  Val events : 190,596  |  Test events : 201,869


## 7. Encoding Helpers & Case Attributes

In [8]:
PM_COLS   = ['case:concept:name', 'concept:name', 'time:timestamp']
SKIP_COLS = {'case:concept:name', 'concept:name', 'time:timestamp', '@@diff_start_end'}
CASE_ATTR_COLS = ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen', 'agency_enc']

case_attrs_train = (
    train_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(train_agency.rename('agency_enc').astype(float))
)
case_attrs_val = (
    val_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(val_agency.rename('agency_enc').astype(float))
)
case_attrs_test = (
    test_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(test_agency.rename('agency_enc').astype(float))
)


def bool_encoding(prefix_pm):
    enriched = pm.extract_outcome_enriched_dataframe(
        prefix_pm,
        activity_key='concept:name',
        timestamp_key='time:timestamp',
        case_id_key='case:concept:name',
    )
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return (
        enriched
        .drop_duplicates('case:concept:name')
        .set_index('case:concept:name')[feat_cols]
        .sort_index()
    )


def freq_encoding(prefix_pm):
    fea_df = pm.extract_features_dataframe(
        prefix_pm,
        activity_key='concept:name',
        timestamp_key='time:timestamp',
        case_id_key='case:concept:name',
        include_case_id=True,
        count_occurrences=True,
    )
    return fea_df.set_index('case:concept:name').sort_index()


def succ_encoding(prefix_pm):
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())
    df_ = prefix_pm.copy()
    df_['prev_act'] = df_.groupby('case:concept:name')['concept:name'].shift(1)
    df_ = df_.dropna(subset=['prev_act'])
    if df_.empty:
        return pd.DataFrame(index=pd.Index(all_case_ids, name='case:concept:name'))
    df_['bigram'] = df_['prev_act'] + ' >> ' + df_['concept:name']
    bigram_counts = (
        df_.groupby(['case:concept:name', 'bigram'])
        .size().unstack(fill_value=0)
    )
    bigram_counts = bigram_counts.reindex(all_case_ids, fill_value=0)
    bigram_counts.index.name = 'case:concept:name'
    return bigram_counts


def build_Xy(X_pm, case_attrs_split, y_series, train_cols=None):
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    y = y_series.reindex(X.index).fillna(0).values.astype(np.int32)
    return X.values.astype(np.float32), y, list(X.columns)


def apply_smotenc(X, y, feat_names, random_state=42):
    if not USE_SMOTENC:
        return X, y
    CASE_ATTRS = {'Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen'}
    cat_mask = np.array([
        name.startswith('concept:name_') or name in CASE_ATTRS
        for name in feat_names
    ])
    cat_indices = np.where(cat_mask)[0].tolist()
    if len(cat_indices) == 0 or len(cat_indices) == len(feat_names):
        sm = SMOTE(random_state=random_state)
    else:
        sm = SMOTENC(categorical_features=cat_indices, random_state=random_state)
    X_res, y_res = sm.fit_resample(X, y)
    print(f'  SMOTE-NC: {len(y):,} → {len(y_res):,} samples '
          f'(minority: {int(y.sum()):,} → {int(y_res.sum()):,})')
    return X_res.astype(np.float32), y_res.astype(np.int32)


print('Helpers defined.')
print(f'Case attributes : {CASE_ATTR_COLS}')

Helpers defined.
Case attributes : ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen', 'agency_enc']


## 8. Full-Trace Feature Extraction (k = all)

In [9]:
full_train_pm = train_df[PM_COLS].copy()
full_val_pm   = val_df[PM_COLS].copy()
full_test_pm  = test_df[PM_COLS].copy()

print('Extracting boolean features...')
X_bool_tr_pm = bool_encoding(full_train_pm)
bool_train_cols = list(X_bool_tr_pm.columns)
X_bool_va_pm = bool_encoding(full_val_pm)
X_bool_te_pm = bool_encoding(full_test_pm)

print('Extracting frequency features...')
X_freq_tr_pm = freq_encoding(full_train_pm)
freq_train_cols = list(X_freq_tr_pm.columns)
X_freq_va_pm = freq_encoding(full_val_pm)
X_freq_te_pm = freq_encoding(full_test_pm)

print('Extracting succession features...')
X_succ_tr_pm = succ_encoding(full_train_pm)
succ_train_cols = list(X_succ_tr_pm.columns)
X_succ_va_pm = succ_encoding(full_val_pm)
X_succ_te_pm = succ_encoding(full_test_pm)

print('Assembling Xy matrices...')
X_bool_tr, y_tr, bool_feat_names = build_Xy(X_bool_tr_pm, case_attrs_train, y_train_series)
X_bool_va, y_va, _               = build_Xy(X_bool_va_pm, case_attrs_val,   y_val_series,   train_cols=bool_train_cols)
X_bool_te, y_te, _               = build_Xy(X_bool_te_pm, case_attrs_test,  y_test_series,  train_cols=bool_train_cols)

X_freq_tr, _,    freq_feat_names = build_Xy(X_freq_tr_pm, case_attrs_train, y_train_series)
X_freq_va, _,    _               = build_Xy(X_freq_va_pm, case_attrs_val,   y_val_series,   train_cols=freq_train_cols)
X_freq_te, _,    _               = build_Xy(X_freq_te_pm, case_attrs_test,  y_test_series,  train_cols=freq_train_cols)

X_succ_tr, _,    succ_feat_names = build_Xy(X_succ_tr_pm, case_attrs_train, y_train_series)
X_succ_va, _,    _               = build_Xy(X_succ_va_pm, case_attrs_val,   y_val_series,   train_cols=succ_train_cols)
X_succ_te, _,    _               = build_Xy(X_succ_te_pm, case_attrs_test,  y_test_series,  train_cols=succ_train_cols)

print('\nApplying SMOTE-NC to training splits...')
print('[Boolean]')
X_bool_tr, y_tr_bool = apply_smotenc(X_bool_tr, y_tr, bool_feat_names,  random_state=RANDOM_STATE)
print('[Frequency]')
X_freq_tr, y_tr_freq = apply_smotenc(X_freq_tr, y_tr, freq_feat_names,  random_state=RANDOM_STATE)
print('[Succession]')
X_succ_tr, y_tr_succ = apply_smotenc(X_succ_tr, y_tr, succ_feat_names,  random_state=RANDOM_STATE)

pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
print(f'\nBoolean    : train {X_bool_tr.shape}  |  val {X_bool_va.shape}  |  test {X_bool_te.shape}')
print(f'Frequency  : train {X_freq_tr.shape}  |  val {X_freq_va.shape}  |  test {X_freq_te.shape}')
print(f'Succession : train {X_succ_tr.shape}  |  val {X_succ_va.shape}  |  test {X_succ_te.shape}')
print(f'pos_weight (XGB, original y_tr): {pos_weight:.1f}')

Extracting boolean features...
Extracting frequency features...
Extracting succession features...
Assembling Xy matrices...

Applying SMOTE-NC to training splits...
[Boolean]
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
[Frequency]
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
[Succession]
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)

Boolean    : train (316424, 47)  |  val (60108, 47)  |  test (57035, 47)
Frequency  : train (316424, 42)  |  val (60108, 42)  |  test (57035, 42)
Succession : train (316424, 550)  |  val (60108, 550)  |  test (57035, 550)
pos_weight (XGB, original y_tr): 40.4


## 9. Train & Evaluate Models

In [10]:
def run_models(X_train, X_val, X_test, y_train, y_val, y_test, encoding_label):
    """Train RF / XGB / LGBM on full traces and report F1, ROC-AUC, PR-AUC."""
    if USE_SMOTENC:
        rf_class_weight   = None
        xgb_spw           = 1.0
        lgbm_class_weight = None
    else:
        rf_class_weight   = 'balanced'
        xgb_spw           = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        lgbm_class_weight = 'balanced'

    models = {
        'RF': RandomForestClassifier(
            n_estimators=500, class_weight=rf_class_weight,
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
        'XGBoost': XGBClassifier(
            n_estimators=500, scale_pos_weight=xgb_spw,
            n_jobs=-1, random_state=RANDOM_STATE,
            verbosity=0, eval_metric='logloss',
            early_stopping_rounds=20,
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=500, class_weight=lgbm_class_weight,
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
        ),
    }

    results, probas, trained = {}, {}, {}
    for name, clf in models.items():
        print(f'  [{encoding_label}] {name}...')
        if name == 'XGBoost':
            clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        elif name == 'LightGBM':
            clf.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)],
            )
        else:
            clf.fit(X_train, y_train)

        proba = clf.predict_proba(X_test)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        results[name] = {
            'Accuracy' : accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred, zero_division=0),
            'Recall'   : recall_score(y_test, pred, zero_division=0),
            'F1'       : f1_score(y_test, pred, zero_division=0),
            'ROC-AUC'  : roc_auc_score(y_test, proba),
            'PR-AUC'   : average_precision_score(y_test, proba),
        }
        probas[name]  = proba
        trained[name] = clf
        m = results[name]
        print(f'    F1={m["F1"]:.4f}  ROC-AUC={m["ROC-AUC"]:.4f}  PR-AUC={m["PR-AUC"]:.4f}')
    return results, probas, trained


print('=== Boolean Encoding (k=all) ===')
bool_results, bool_probas, bool_models = run_models(
    X_bool_tr, X_bool_va, X_bool_te, y_tr_bool, y_va, y_te, 'Boolean'
)

print('\n=== Frequency Encoding (k=all) ===')
freq_results, freq_probas, freq_models = run_models(
    X_freq_tr, X_freq_va, X_freq_te, y_tr_freq, y_va, y_te, 'Frequency'
)

print('\n=== Succession Encoding (k=all) ===')
succ_results, succ_probas, succ_models = run_models(
    X_succ_tr, X_succ_va, X_succ_te, y_tr_succ, y_va, y_te, 'Succession'
)

=== Boolean Encoding (k=all) ===
  [Boolean] RF...
    F1=0.8494  ROC-AUC=0.9899  PR-AUC=0.8402
  [Boolean] XGBoost...
    F1=0.8098  ROC-AUC=0.9967  PR-AUC=0.8288
  [Boolean] LightGBM...
    F1=0.8264  ROC-AUC=0.9971  PR-AUC=0.8440

=== Frequency Encoding (k=all) ===
  [Frequency] RF...
    F1=0.7895  ROC-AUC=0.9870  PR-AUC=0.7929
  [Frequency] XGBoost...
    F1=0.7860  ROC-AUC=0.9852  PR-AUC=0.8194
  [Frequency] LightGBM...
    F1=0.7106  ROC-AUC=0.9832  PR-AUC=0.8205

=== Succession Encoding (k=all) ===
  [Succession] RF...
    F1=0.8463  ROC-AUC=0.9767  PR-AUC=0.8503
  [Succession] XGBoost...
    F1=0.8016  ROC-AUC=0.9793  PR-AUC=0.8343
  [Succession] LightGBM...
    F1=0.8060  ROC-AUC=0.9791  PR-AUC=0.8428


## 10. Results Summary

In [11]:
rows = []
for encoding, results in [('Boolean', bool_results), ('Frequency', freq_results), ('Succession', succ_results)]:
    for model, metrics in results.items():
        rows.append({'Encoding': encoding, 'Model': model, **metrics})

summary = pd.DataFrame(rows).set_index(['Encoding', 'Model'])
print('\n=== Full-Trace Results (k=all) — Test Set ===')
print(summary[['F1', 'ROC-AUC', 'PR-AUC', 'Precision', 'Recall', 'Accuracy']].to_string(float_format='{:.4f}'.format))


=== Full-Trace Results (k=all) — Test Set ===
                        F1  ROC-AUC  PR-AUC  Precision  Recall  Accuracy
Encoding   Model                                                        
Boolean    RF       0.8494   0.9899  0.8402     0.8146  0.8874    0.9962
           XGBoost  0.8098   0.9967  0.8288     0.7830  0.8384    0.9952
           LightGBM 0.8264   0.9971  0.8440     0.7822  0.8759    0.9955
Frequency  RF       0.7895   0.9870  0.7929     0.7074  0.8932    0.9942
           XGBoost  0.7860   0.9852  0.8194     0.7035  0.8903    0.9941
           LightGBM 0.7106   0.9832  0.8205     0.6247  0.8240    0.9918
Succession RF       0.8463   0.9767  0.8503     0.8136  0.8817    0.9961
           XGBoost  0.8016   0.9793  0.8343     0.7529  0.8571    0.9948
           LightGBM 0.8060   0.9791  0.8428     0.7652  0.8514    0.9950


In [12]:
best_idx = summary['ROC-AUC'].idxmax()
best     = summary.loc[best_idx]
print(f'Best model (ROC-AUC): {best_idx}  →  ROC-AUC={best["ROC-AUC"]:.4f}  PR-AUC={best["PR-AUC"]:.4f}  F1={best["F1"]:.4f}')

Best model (ROC-AUC): ('Boolean', 'LightGBM')  →  ROC-AUC=0.9971  PR-AUC=0.8440  F1=0.8264


## 11. PR Curves (Best Encoding per Model)

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (enc_label, probas_dict, y_test) in zip(
    axes,
    [('Boolean', bool_probas, y_te),
     ('Frequency', freq_probas, y_te),
     ('Succession', succ_probas, y_te)],
):
    for model_name, proba in probas_dict.items():
        p, r, _ = precision_recall_curve(y_test, proba)
        prauc = average_precision_score(y_test, proba)
        ax.plot(r, p, label=f'{model_name} (PR-AUC={prauc:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{enc_label} Encoding')
    ax.legend(fontsize=8)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

fig.suptitle('PR Curves — Full Trace (k=all)', fontsize=13)
plt.tight_layout()
plt.savefig('figs/pr_curves_outcome_fulltrace.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figs/pr_curves_outcome_fulltrace.png')

Saved: figs/pr_curves_outcome_fulltrace.png
